# Classroom Seating Optimizer

This notebook mirrors the desktop application's use case, but uses an editable input block instead of `input()` prompts. That makes it reliable in VS Code notebooks: edit the values in one place, then run the cells from top to bottom.

Synthetic student data is enabled by default to reduce repetitive data entry. It is clearly labelled and is only for demonstrating the application; use the Excel/CSV import or replace the manual list for a real class.

In [1]:
# ===================== EDITABLE INPUTS =====================
# These correspond to the fields in the desktop application.
USE_SYNTHETIC_STUDENTS = True
STUDENT_COUNT = 30
RANDOM_SEED = 1

# For real/manual data, set USE_SYNTHETIC_STUDENTS = False and edit this list.
# Format: (ID, first name, last name, major, GPA, year)
MANUAL_STUDENTS = [
    ('AU2320001', 'Aarav', 'Patel', 'CSE', 6.4, '1'),
    ('AU2320002', 'Diya', 'Shah', 'ECE', 8.2, '1'),
]

ROWS = 6
COLUMNS = 9
UNAVAILABLE_SEATS = {(1, 9), (6, 1)}

# Hard restrictions. IDs must exist in the student records.
SEPARATED_PAIRS = []                 # e.g. [('AU2320001', 'AU2320002')]
FIXED_SEATS = {}                     # e.g. {'AU2320001': (2, 3)}
HARD_FRONT_RULE = False

# Soft preferences. Lower total soft cost is better.
FRONT_ROWS = 2
LOW_GPA_THRESHOLD = 7.0
PREFER_LOW_GPA_IN_FRONT = True
SPREAD_MAJORS = True
FRONT_WEIGHT = 3
MAJOR_WEIGHT = 1
SAVE_HTML_REPORT = False

print('Inputs loaded. Synthetic data:', USE_SYNTHETIC_STUDENTS)

Inputs loaded. Synthetic data: True


## What the inputs mean

The project is a **Constraint Satisfaction Problem (CSP)**. Students are variables and available coordinates are their domains. Hard restrictions must always hold: seats are all-different, unavailable seats cannot be used, fixed seats stay fixed, separated pairs cannot sit beside each other, and an optional front-row rule can be hard.

Soft preferences are not treated as absolute rules. They contribute to a weighted cost: front-row preference, major spreading, and (in the desktop rearrangement workflow) movement from a previous chart. This distinction lets the solver report infeasibility honestly.

In [2]:
# Static code form of the application's core configuration.
print('config = Config(rows, cols, unavailable=..., separate=..., fixed=...,')
print('                front_rows=..., cgpa_threshold=..., spread_majors=...)')
print('result = SeatingProblem(students, config).solve()')

config = Config(rows, cols, unavailable=..., separate=..., fixed=...,
                front_rows=..., cgpa_threshold=..., spread_majors=...)
result = SeatingProblem(students, config).solve()


## Generate student records and run the arrangement

The next cell is intentionally self-contained. It validates IDs and coordinates before search, creates synthetic data only when requested, runs backtracking with MRV, degree tie-breaking, forward checking, and soft-cost value ordering, then applies local hill climbing.

In [3]:
from html import escape
from pathlib import Path
from IPython.display import HTML, display
from data import Student, generate_students
from report import alphabetical_assignment, comparison_summary, text_grid, validation_summary, write_html
from seating import Config, SeatingProblem

if USE_SYNTHETIC_STUDENTS:
    students = generate_students(STUDENT_COUNT, seed=RANDOM_SEED)
    print(f'Synthetic data generated: {len(students)} records, seed={RANDOM_SEED}.')
else:
    students = [Student(sid, f'{first} {last}', major, gpa, year) for sid, first, last, major, gpa, year in MANUAL_STUDENTS]
    print(f'Manual data loaded: {len(students)} records.')

ids = [student.id for student in students]
if len(set(ids)) != len(ids):
    raise ValueError('Student IDs must be unique.')
if not students:
    raise ValueError('Add at least one student.')
if len(students) > ROWS * COLUMNS - len(UNAVAILABLE_SEATS):
    raise ValueError('There are more students than available seats.')
if any(row < 1 or row > ROWS or col < 1 or col > COLUMNS for row, col in UNAVAILABLE_SEATS):
    raise ValueError('Every unavailable seat must be inside the classroom layout.')
referenced_ids = {sid for pair in SEPARATED_PAIRS for sid in pair} | set(FIXED_SEATS)
unknown_ids = referenced_ids - set(ids)
if unknown_ids:
    raise ValueError(f'Constraint references unknown IDs: {sorted(unknown_ids)}')

config = Config(ROWS, COLUMNS, unavailable=UNAVAILABLE_SEATS, separate=SEPARATED_PAIRS,
                fixed=FIXED_SEATS, front_rows=FRONT_ROWS,
                cgpa_threshold=LOW_GPA_THRESHOLD,
                spread_majors=SPREAD_MAJORS,
                w_front=FRONT_WEIGHT if PREFER_LOW_GPA_IN_FRONT else 0,
                w_major=MAJOR_WEIGHT, hard_front=HARD_FRONT_RULE)
problem = SeatingProblem(students, config)
result = problem.solve()

if result.assignment is None:
    raise RuntimeError(result.message)

student_rows = ['<table><tr><th>ID</th><th>Name</th><th>Major</th><th>GPA</th><th>Year</th></tr>']
for student in students:
    student_rows.append(f'<tr><td>{escape(student.id)}</td><td>{escape(student.name)}</td><td>{escape(student.major)}</td><td>{student.cgpa:.2f}</td><td>{escape(str(student.year))}</td></tr>')
student_rows.append('</table>')
display(HTML(''.join(student_rows)))
print(text_grid(problem, result.assignment))
print(validation_summary(problem, result.assignment))
print(f'Search nodes: {result.stats.nodes} | Backtracks: {result.stats.backtracks}')

Synthetic data generated: 30 records, seed=1.


ID,Name,Major,GPA,Year
AU2320001,Karan Parikh,MECH,8.58,
AU2320002,Isha Gupta,BBA,6.21,
AU2320003,Hetal Sharma,CSE,7.71,
AU2320004,Vivek Sharma,CSE,7.17,
AU2320005,Smit Gupta,CSE,7.42,
AU2320006,Tanvi Patel,CSE,9.39,
AU2320007,Aarav Modi,ECE,9.66,
AU2320008,Anaya Patel,ECE,7.72,
AU2320009,Dev Gupta,ECE,6.73,
AU2320010,Dev Joshi,CSE,7.30,


            Seat1  Seat2  Seat3  Seat4  Seat5  Seat6  Seat7  Seat8  Seat9  
  [FRONT / BOARD]
Row 1   002*   016*   009*   023*   024    001    006    ...    XXX    
Row 2   027*   019*   004    017    029    ...    ...    ...    ...    
Row 3   013    018    010    015    021    ...    ...    ...    ...    
Row 4   012    022    005    008    007    ...    ...    ...    ...    
Row 5   028    003    030    025    026    ...    ...    ...    ...    
Row 6   XXX    020    011    014    ...    ...    ...    ...    ...    
(cell = last 3 digits of enrollment ID; * = CGPA below threshold; XXX = unavailable; ... = empty)
Hard constraints: ALL SATISFIED
Front-row preference: 6/6 low-CGPA students in first 2 rows
Major spread: 7 same-major pairs in a row (best possible about 7)
Weighted soft cost: 7  |  Soft satisfaction score: 100.0/100
Course theory: CSP: students are variables and classroom seats are domains. Backtracking searches for a consistent assignment. MRV (Minimum Remaining Values)

## Alphabetical baseline and grounded comparison

The baseline uses the same students, available seats, and fixed seats, then assigns the remaining students alphabetically by name. It is not repaired to satisfy the other rules. The comparison is grounded in actual measurements: hard violations, low-GPA front placement, same-major pairs in a row, and weighted soft cost.

MRV means **Minimum Remaining Values** and follows the fail-first principle. The degree heuristic breaks ties using constraint connectivity. Forward checking prunes impossible values after each assignment. These search techniques explain why the generated chart can satisfy constraints that alphabetical placement may miss.

In [4]:
alphabetical = alphabetical_assignment(problem)
print(comparison_summary(problem, result.assignment, alphabetical))

if SAVE_HTML_REPORT:
    report_path = Path('notebook_seating_report.html')
    write_html(report_path, problem, {}, result.assignment, [], baseline=alphabetical)
    print(f'Report saved to {report_path.resolve()}')
    display(HTML(report_path.read_text(encoding='utf-8')))
else:
    print('Set SAVE_HTML_REPORT = True in the input block to save and display the HTML report.')

Alphabetical baseline comparison (same students and classroom):
- Hard violations: optimized 0 vs alphabetical 0
- Low-GPA students in front rows: optimized 6/6 vs alphabetical 3/6
- Same-major pairs in a row: optimized 7 vs alphabetical 32 (lower is better)
- Weighted soft cost: optimized 7 vs alphabetical 41 (lower is better)
Conclusion: the generated arrangement is better for the selected soft preferences because its measured weighted cost is lower.
Set SAVE_HTML_REPORT = True in the input block to save and display the HTML report.
